# HEART — Random Duplicate-Impact Test (READ ONLY)

**Question:** Do duplicate molecular identities make the original Uni-Mol ligand sampling substantially different from uniform sampling of unique molecules?

- Original training LMDB: **18,837,320** records; completed raw-SMILES audit: **13,270,476** distinct strings.
- **Random sample only:** 1,500 raw LMDB records + 1,500 independently sampled unique identities from the **existing full-audit hash index**.
- Retrieve the **exact full-dataset copy counts** for these sampled identities by streaming the previously exported ~620 MB digest file once. **Do not rescan the large LMDB, do not retrain, no Slurm, no checkpoint/GPU required.**
- Compare raw-record-weighted vs unique-identity-weighted duplicate-frequency distributions. A small raw-record sample also gives a **rough, reweighted** RDKit chemical-property comparison.
- **This does NOT measure downstream model accuracy improvement from deduplicated training.** That requires a matched-training comparison; here we decide whether such an experiment is justified.
- Raw-SMILES identity only; canonical SMILES or tautomer/stereochemical policies require a separate analysis.

## 0 — Paths and safety checks

In [ ]:
from pathlib import Path
import json, hashlib, pickle, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 17
N_RAW = 1500
N_UNIQUE = 1500

HOME = Path('/home/ypan/heart_liver/model_train')
PROJECT = Path('/project/yli4/ypan')
SCRATCH = Path('/scratch/ypan/heart_liver/model_train')

# Override if your paths differ; notebook does not search the entire filesystem.
DATA_OVERRIDE = None
AUDIT_OVERRIDE = None
RESULTS_OVERRIDE = None

DATA_CANDIDATES = [
    PROJECT/'UniMolReference/ligand_data/ligands',
    HOME/'UniMolReference/ligand_data/ligands',
    SCRATCH/'UniMolReference/ligand_data/ligands',
]
AUDIT_CANDIDATES = [
    HOME/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
    PROJECT/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
    SCRATCH/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
]

DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else next(
    (p for p in DATA_CANDIDATES if (p/'train.lmdb').is_file()), DATA_CANDIDATES[0])
AUDIT = Path(AUDIT_OVERRIDE) if AUDIT_OVERRIDE else next(
    (p for p in AUDIT_CANDIDATES if (p/'train_raw_hashes.txt').is_file()), AUDIT_CANDIDATES[0])
RESULTS = (Path(RESULTS_OVERRIDE) if RESULTS_OVERRIDE else
           HOME/'heart_evaluation/duplicate_impact_quick_test')

TRAIN_LMDB = DATA/'train.lmdb'
RAW_HASHES = AUDIT/'train_raw_hashes.txt'
UNIQUE_HASHES = AUDIT/'train_raw_unique_sorted.txt'
SUMMARY = AUDIT/'audit_summary_raw.json'
EXPORT = AUDIT/'train_raw_export.json'

for name,p in [('train LMDB',TRAIN_LMDB),('raw hash export',RAW_HASHES),
               ('unique sorted hashes',UNIQUE_HASHES),('audit summary',SUMMARY),('audit provenance',EXPORT)]:
    print(f'{name:22} exists={p.is_file()}  {p}')
missing=[p for p in (TRAIN_LMDB,RAW_HASHES,UNIQUE_HASHES,SUMMARY,EXPORT) if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing existing audit assets: '+', '.join(map(str,missing))+
                            '. Do not rerun 18.8M audit; locate its existing output and set AUDIT_OVERRIDE above.')

full=json.loads(SUMMARY.read_text())
ex=json.loads(EXPORT.read_text())
assert full['method']=='raw' and not full['train']['limited'], 'Requires COMPLETED full raw audit'
assert ex['identity']=='raw' and not ex['limited'], 'This is a partial or non-raw hash export'
assert full['train']['total_records_scanned']==18_837_320, 'Unexpected training dataset size'
assert int(ex['usable_records'])==full['train']['usable_records']
expected_path=Path(ex['path']).resolve()
if expected_path!=TRAIN_LMDB.resolve():
    raise RuntimeError(f'Audit hashes refer to {expected_path}, but current DATA is {TRAIN_LMDB.resolve()}. '
                       'Do not mix different LMDB versions. Restore the matching source or prove file equivalence first.')
assert RAW_HASHES.stat().st_size==33*ex['usable_records'], 'Raw-hash file is incomplete'
assert UNIQUE_HASHES.stat().st_size==33*full['train']['unique_identifiers'], 'Unique-hash file is incomplete'
print('\nFull audit verified: total',f"{full['train']['total_records_scanned']:,}",
      'raw-unique',f"{full['train']['unique_identifiers']:,}",
      'duplicate excess',f"{full['train']['duplicate_record_excess']:,}")
RESULTS.mkdir(parents=True,exist_ok=True)
print('NEW outputs only:',RESULTS)

## 1 — Random raw records + random **unique molecule identities**

The independent unique-ID sample comes from `train_raw_unique_sorted.txt`, not from deduplicating a 1.5K mini-sample (which would be misleading when there are 13M identities).

In [ ]:
import lmdb
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
RDLogger.DisableLog('rdApp.error')

def hash_smi(s):
    return hashlib.blake2b(s.strip().encode('utf8'),digest_size=16).hexdigest()

# The hashed file is a fixed-width ASCII table: 32-hex-digit digest plus newline.
# Seek by exact line offset, so sampling unique molecules does not load 13M hashes into RAM.
rng=np.random.default_rng(SEED)
unique_n=int(full['train']['unique_identifiers'])
unique_line_idx=np.sort(rng.choice(unique_n,size=min(N_UNIQUE,unique_n),replace=False))
unique_hashes=[]
with UNIQUE_HASHES.open('rb') as f:
    for i in unique_line_idx:
        f.seek(33*int(i))
        entry=f.read(33)
        if len(entry)!=33 or entry[32:33]!=b'\n':
            raise ValueError('Unexpected hash index format')
        unique_hashes.append(entry[:32].decode('ascii'))
assert len(set(unique_hashes))==len(unique_hashes)

# Actual raw records: sample numeric LMDB keys 0..N-1 (NOT cursor ordinal positions).
env=lmdb.open(str(TRAIN_LMDB),subdir=False,readonly=True,lock=False,
              readahead=False,meminit=False,max_readers=128)
raw_rows=[]
try:
    with env.begin(write=False) as txn:
        n_lmdb=txn.stat()['entries']
        assert n_lmdb==full['train']['total_records_scanned'], (n_lmdb,full['train']['total_records_scanned'])
        raw_keys=rng.choice(n_lmdb,size=min(N_RAW,n_lmdb),replace=False)
        for j,k in enumerate(raw_keys,1):
            v=txn.get(str(int(k)).encode('ascii'))
            if v is None:raise KeyError(f'LMDB numeric key missing: {k}')
            d=pickle.loads(v)  # Trusted local Uni-Mol LMDB; read-only.
            smi=d.get('smi',d.get(b'smi'))
            if isinstance(smi,bytes):smi=smi.decode('utf-8')
            if not isinstance(smi,str) or not smi.strip():
                continue
            smi=smi.strip()
            mol=Chem.MolFromSmiles(smi)
            raw_rows.append(dict(
                lmdb_key=int(k),smiles=smi,digest=hash_smi(smi),
                conformer_slots=len(d.get('coordinates',[])),
                heavy_atoms=mol.GetNumHeavyAtoms() if mol else np.nan,
                molecular_weight=Descriptors.MolWt(mol) if mol else np.nan,
                logP=Descriptors.MolLogP(mol) if mol else np.nan,
                TPSA=rdMolDescriptors.CalcTPSA(mol) if mol else np.nan,
            ))
            if j%500==0:print('Random numeric LMDB records sampled:',j,'/',len(raw_keys),flush=True)
finally:
    env.close()
raw=pd.DataFrame(raw_rows)
if raw.empty:raise RuntimeError('No valid SMILES in sampled records')
print('Raw records sampled:',len(raw),'; separately sampled distinct hashed molecules:',len(unique_hashes))
print('Conformer slots per record (raw sample):',raw.conformer_slots.value_counts().head().to_dict())

## 2 — Count global occurrence for the sampled molecules

One **sequential pass over the existing 18.84M-line hash text export**, not over the full LMDB. Reports exact full-data copy count for each randomly chosen molecule.

In [ ]:
targets=set(raw['digest'])|set(unique_hashes)
freq={key:0 for key in targets}
t0=time.monotonic()
scanned=0
with RAW_HASHES.open('rb',buffering=8*1024*1024) as f:
    for line in f:
        scanned+=1
        h=line[:32].decode('ascii')
        if h in freq:freq[h]+=1
        if scanned%5_000_000==0:
            print(f'Existing hashes scanned {scanned:,} / {ex["usable_records"]:,}',flush=True)
assert scanned==int(ex['usable_records']), 'Raw audit export truncated'
assert min(freq.values())>0, 'A selected digest was not found. Check path & audit provenance.'
raw['copies_in_full_train']=raw['digest'].map(freq).astype('int64')
uniq=pd.DataFrame({'digest':unique_hashes})
uniq['copies_in_full_train']=uniq['digest'].map(freq).astype('int64')
print('Sampled frequencies ready; elapsed',round(time.monotonic()-t0,1),'s')
print('Raw-record sample: median copies',raw.copies_in_full_train.median(),
      '; fraction belonging to repeated identities',round((raw.copies_in_full_train>1).mean()*100,2),'%')
print('Deduplicated-identity sample: median copies in original file',uniq.copies_in_full_train.median(),
      '; fraction of unique identities repeated in original',round((uniq.copies_in_full_train>1).mean()*100,2),'%')

## 3 — Raw vs deduplicated: summary and figures

**Interpretation:** raw sampling is proportional to the number of rows per SMILES; deduplicated sampling gives each distinct raw SMILES equal probability. The same molecules are not being re-inferred by HEART here.

In [ ]:
def binned(v):
    b=[0,1,2,3,5,10,20,np.inf]
    labels=['1','2','3','4–5','6–10','11–20','>20']
    return pd.cut(v,bins=b,labels=labels,include_lowest=True)
raw_hist=binned(raw.copies_in_full_train).value_counts(normalize=True).sort_index()
uniq_hist=binned(uniq.copies_in_full_train).value_counts(normalize=True).sort_index()
comparison=pd.DataFrame({'Raw record sampling (%)':100*raw_hist,
                         'Dedup unique molecule sampling (%)':100*uniq_hist}).fillna(0)
print('Copy-count distribution — full dataset frequency for sampled molecules:')
display(comparison.round(2))
summary=pd.DataFrame([
    dict(sampling='Raw random records',sample_n=len(raw),
         pct_repeat_identity=100*(raw.copies_in_full_train>1).mean(),
         median_full_copy_count=raw.copies_in_full_train.median(),
         p95_full_copy_count=raw.copies_in_full_train.quantile(.95),
         max_full_copy_count_in_sample=raw.copies_in_full_train.max()),
    dict(sampling='Uniform dedup unique molecules',sample_n=len(uniq),
         pct_repeat_identity=100*(uniq.copies_in_full_train>1).mean(),
         median_full_copy_count=uniq.copies_in_full_train.median(),
         p95_full_copy_count=uniq.copies_in_full_train.quantile(.95),
         max_full_copy_count_in_sample=uniq.copies_in_full_train.max()),
])
display(summary.round(2))

fig,ax=plt.subplots(figsize=(9,4.5))
pos=np.arange(len(comparison))
ax.bar(pos-.19,comparison['Raw record sampling (%)'].to_numpy(),.38,label='Raw record sampling')
ax.bar(pos+.19,comparison['Dedup unique molecule sampling (%)'].to_numpy(),.38,label='Uniform unique molecules')
ax.set_xticks(pos,comparison.index.astype(str));ax.set_xlabel('Number of records per raw SMILES in FULL training LMDB')
ax.set_ylabel('Share of sampled observations (%)')
ax.set_title('How duplication changes molecular exposure')
ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout()
fig.savefig(RESULTS/'01_raw_vs_dedup_copy_frequency.png',dpi=190)
fig.savefig(RESULTS/'01_raw_vs_dedup_copy_frequency.pdf')
plt.show()

summary.to_csv(RESULTS/'frequency_sample_summary.csv',index=False)
comparison.to_csv(RESULTS/'frequency_histogram.csv')
raw.to_csv(RESULTS/'random_raw_records_with_global_frequency.csv',index=False)
uniq.to_csv(RESULTS/'random_unique_hashes_with_global_frequency.csv',index=False)
print('Output:',RESULTS)

## 4 — Optional quick chemistry check: how raw vs uniform-unique exposure changes sampled descriptors

Use the randomly sampled **raw records**, with inverse-frequency weights `1 / copies_in_full_train` as a rough importance-weighted estimate of uniform-unique sampling. This is **not** a second uniform sample of SMILES and has sampling uncertainty. It is intended for quick screening only.

In [ ]:
properties=['heavy_atoms','molecular_weight','logP','TPSA']
rows=[]
w=1.0/raw.copies_in_full_train.to_numpy(dtype=float)
for name in properties:
    x=raw[name].to_numpy(dtype=float)
    ok=np.isfinite(x)
    if not ok.any():continue
    raw_mean=float(np.mean(x[ok]))
    dedup_mean_est=float(np.average(x[ok],weights=w[ok]))
    rows.append(dict(property=name,raw_record_mean=raw_mean,
                     dedup_unique_mean_est=dedup_mean_est,
                     difference=dedup_mean_est-raw_mean,n_usable=int(ok.sum())))
chemical=pd.DataFrame(rows)
display(chemical.round(3))
chemical.to_csv(RESULTS/'chemical_descriptor_reweight_preview.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,feature in zip(axes,['heavy_atoms','logP']):
    x=raw[feature].to_numpy(dtype=float)
    ok=np.isfinite(x)
    if not ok.any():continue
    lo,hi=np.quantile(x[ok],[.01,.99]);bins=np.linspace(lo,hi,21)
    ax.hist(x[ok],bins=bins,weights=np.ones(ok.sum())/ok.sum(),
            alpha=.5,label='Raw records')
    weights=w[ok]/w[ok].sum()
    ax.hist(x[ok],bins=bins,weights=weights,alpha=.5,
            label='Estimated uniform unique')
    ax.set(title=f'Sample-weighted {feature} distribution',xlabel=feature,ylabel='Proportion')
    ax.legend()
plt.tight_layout()
fig.savefig(RESULTS/'02_chemistry_raw_vs_dedup_preview.png',dpi=190)
fig.savefig(RESULTS/'02_chemistry_raw_vs_dedup_preview.pdf')
plt.show()

report={
    'purpose':'Random duplicate-impact comparison; NO model training or model accuracy evaluation',
    'identity':'raw SMILES (BLAKE2b-128)',
    'full_train_records':int(full['train']['total_records_scanned']),
    'full_train_unique_raw_smiles':int(full['train']['unique_identifiers']),
    'full_duplicate_record_excess':int(full['train']['duplicate_record_excess']),
    'full_duplicate_record_excess_fraction':float(full['train']['duplicate_record_excess_fraction']),
    'n_raw_random_records':len(raw),'n_independent_unique_hashes':len(uniq),'random_seed':SEED,
    'raw_sample_fraction_of_records_with_repeated_identity':float((raw.copies_in_full_train>1).mean()),
    'unique_sample_fraction_of_identifiers_repeated_in_original':float((uniq.copies_in_full_train>1).mean()),
    'caveat':'Same-SMILES records can contain different conformers; this test does not measure their independent conformer diversity, nor the impact of retraining on deduplicated data.'
}
(RESULTS/'report.json').write_text(json.dumps(report,indent=2)+'\n')
print('SAVED: plots, CSV, and report.json to',RESULTS)
print('IMPORTANT: If these distributions differ strongly, it supports a balanced-sampling ablation,')
print('but it does NOT prove that a retrained deduplicated HEART will outperform the current checkpoint.')

## Reading the results

- **Figure 01 is the primary outcome:** a strong gap between the blue/orange bars means record-weighted training disproportionately exposes the model to multiply recorded molecules.
- **Figure 02 is exploratory:** descriptor shifts may suggest chemical-frequency bias; treat apparent differences cautiously because the 1.5K sample is small.
- `report.json`: exact *full-population* duplicate-excess fraction, plus sampled molecule-copy statistics.
- **No original LMDB is modified.** No deduplicated LMDB is written. No train/validation repartition or benchmark evaluation is performed.
- This test establishes whether duplicate *sampling frequency* merits a controlled retraining ablation; it cannot on its own decide whether downstream FreeSolv/ESOL/Lipophilicity prediction would improve.